# توقع مغادرة عملاء البنك

نقرأ بيانات العملاء، ثم نقسمها إلى تدريب واختبار قبل موازنة الفئات أو تعلم أي تحويل. ندرّب شبكة عصبية بطبقتين مخفيتين من 6 وحدات لكل طبقة، ونقيّمها على بيانات الاختبار الأصلية.

**المتطلبات:** `pandas` و`numpy` و`scikit-learn` و`imbalanced-learn` و`tensorflow`، بالإضافة إلى ملف `Churn_Modelling.csv`.

## 1. استيراد المكتبات وضبط العشوائية

نثبت البذور لتحسين قابلية إعادة النتائج. قد تبقى فروق حسابية طفيفة بين الأجهزة أو إصدارات المكتبات.

In [9]:
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from imblearn.over_sampling import RandomOverSampler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential
from sklearn.metrics import accuracy_score,precision_score,f1_score,recall_score

np.random.seed(42)
tf.random.set_seed(42)


## 2. قراءة البيانات وفحصها

يبحث الدفتر عن الملف في مسارات شائعة، ومنها مسار المشروع الأصلي. عند نقله إلى جهاز آخر، ضع ملف CSV بجوار الدفتر أو عدّل قائمة المسارات. استدعاء `df.info()` يعرض بنية الجدول فعلًا.

In [2]:
dataset_candidates = [
    Path('Churn_Modelling.csv'),
    Path('datasets/Churn_Modelling.csv'),
    Path('../datasets/Churn_Modelling.csv'),
    Path('/home/salim/Projects/ai-learning-journey/datasets/Churn_Modelling.csv'),
]
dataset_path = next((path for path in dataset_candidates if path.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError('ضع Churn_Modelling.csv بجوار الدفتر أو عدّل dataset_candidates.')

df = pd.read_csv(dataset_path)
print('الأعمدة:', df.columns.tolist())
df.info()
print()
print('توزيع الفئة الهدف:')
print(df['Exited'].value_counts())
df.head()


الأعمدة: ['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 3. اختيار المدخلات وتقسيم البيانات

نستبعد أعمدة التعريف (`RowNumber` و`CustomerId` و`Surname`) لأنها ليست خصائص سلوكية مفيدة للنموذج. نستخدم تقسيمًا طبقيًا يحافظ تقريبًا على نسبة فئتي `Exited` في التدريب والاختبار. يحدث هذا **قبل** الموازنة والتحويلات لمنع تسرب معلومات الاختبار.

In [3]:
X = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Exited'])
y = df['Exited']

X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

print('التدريب قبل الموازنة:')
print(y_train_raw.value_counts())
print()
print('الاختبار الأصلي:')
print(y_test.value_counts())


التدريب قبل الموازنة:
Exited
0    5972
1    1528
Name: count, dtype: int64

الاختبار الأصلي:
Exited
0    1991
1     509
Name: count, dtype: int64


## 4. موازنة بيانات التدريب فقط

تكرر `RandomOverSampler` أمثلة الفئة الأقل تمثيلًا في التدريب. لا نمس مجموعة الاختبار حتى تعكس توزيع البيانات الأصلي عند التقييم.

In [4]:
sampler = RandomOverSampler(random_state=0)
X_train_balanced, y_train_balanced = sampler.fit_resample(X_train_raw, y_train_raw)

print('التدريب بعد الموازنة:')
print(y_train_balanced.value_counts())


التدريب بعد الموازنة:
Exited
1    5972
0    5972
Name: count, dtype: int64


## 5. ترميز الفئات وتقييس المدخلات

نتعلم ترميز `Geography` و`Gender` من التدريب المتوازن وحده، ثم نطبقه على الاختبار. بعد ذلك نلائم `StandardScaler` على التدريب المحوّل فقط ونستخدمه لتحويل الاختبار. يتجاهل الترميز الفئات الجديدة في الاختبار إن وجدت.

In [5]:
categorical_columns = ['Geography', 'Gender']
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_columns)
    ],
    remainder='passthrough',
    sparse_threshold=0,
)

X_train_encoded = preprocessor.fit_transform(X_train_balanced)
X_test_encoded = preprocessor.transform(X_test_raw)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_encoded)
X_test = scaler.transform(X_test_encoded)

print('شكل التدريب:', X_train.shape)
print('شكل الاختبار:', X_test.shape)


شكل التدريب: (11944, 13)
شكل الاختبار: (2500, 13)


## 6. بناء النموذج وتدريبه

تتكون الشبكة من طبقتين مخفيتين، في كل منهما 6 وحدات بتفعيل `relu`، ثم مخرج واحد بتفعيل `sigmoid`. نستخدم خسارة التصنيف الثنائي وندرب النموذج لمدة 20 حقبة.

In [11]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(6, activation='relu'),
    Dense(6, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train, y_train_balanced,
    batch_size=10,
    epochs=20,
    verbose=2,
)


Epoch 1/20
1195/1195 - 5s - 4ms/step - accuracy: 0.6458 - loss: 0.6246
Epoch 2/20
1195/1195 - 3s - 3ms/step - accuracy: 0.7360 - loss: 0.5380
Epoch 3/20
1195/1195 - 3s - 3ms/step - accuracy: 0.7589 - loss: 0.5010
Epoch 4/20
1195/1195 - 3s - 2ms/step - accuracy: 0.7642 - loss: 0.4876
Epoch 5/20
1195/1195 - 3s - 3ms/step - accuracy: 0.7650 - loss: 0.4828
Epoch 6/20
1195/1195 - 3s - 3ms/step - accuracy: 0.7670 - loss: 0.4797
Epoch 7/20
1195/1195 - 3s - 2ms/step - accuracy: 0.7682 - loss: 0.4775
Epoch 8/20
1195/1195 - 3s - 3ms/step - accuracy: 0.7688 - loss: 0.4754
Epoch 9/20
1195/1195 - 3s - 2ms/step - accuracy: 0.7694 - loss: 0.4727
Epoch 10/20
1195/1195 - 3s - 2ms/step - accuracy: 0.7711 - loss: 0.4701
Epoch 11/20
1195/1195 - 3s - 3ms/step - accuracy: 0.7730 - loss: 0.4681
Epoch 12/20
1195/1195 - 3s - 2ms/step - accuracy: 0.7751 - loss: 0.4658
Epoch 13/20
1195/1195 - 3s - 3ms/step - accuracy: 0.7757 - loss: 0.4635
Epoch 14/20
1195/1195 - 3s - 2ms/step - accuracy: 0.7766 - loss: 0.4618
E

## 7. التقييم على بيانات الاختبار

نحسب الخسارة والدقة على مجموعة الاختبار التي لم تُستخدم في التدريب أو ملاءمة التحويلات. يوضح تقرير التصنيف أداء كل فئة، وتبين مصفوفة الالتباس عدد التوقعات الصحيحة والخاطئة.

In [8]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'خسارة الاختبار: {test_loss:.4f}')
print(f'دقة الاختبار: {test_accuracy:.4f}')

y_probability = model.predict(X_test, verbose=0)
y_pred = (y_probability >= 0.5)

print()
print('مصفوفة الالتباس (الصفوف: الحقيقة، الأعمدة: التوقع):')
print(confusion_matrix(y_test, y_pred))
print()
print('تقرير التصنيف:')
print(classification_report(y_test, y_pred, digits=3))


خسارة الاختبار: 0.4330
دقة الاختبار: 0.8088

مصفوفة الالتباس (الصفوف: الحقيقة، الأعمدة: التوقع):
[[1662  329]
 [ 149  360]]

تقرير التصنيف:
              precision    recall  f1-score   support

           0      0.918     0.835     0.874      1991
           1      0.522     0.707     0.601       509

    accuracy                          0.809      2500
   macro avg      0.720     0.771     0.738      2500
weighted avg      0.837     0.809     0.819      2500



In [10]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(accuracy, precision, recall, f1, sep='\t')

0.8088	0.5224963715529753	0.7072691552062869	0.6010016694490818


In [26]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train, y_train_balanced,
    batch_size=100,
    epochs=200,
)


Epoch 1/200
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6998 - loss: 0.5714   
Epoch 2/200
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7766 - loss: 0.4604  
Epoch 3/200
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7898 - loss: 0.4371  
Epoch 4/200
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 895us/step - accuracy: 0.7996 - loss: 0.4228
Epoch 5/200
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8075 - loss: 0.4103  
Epoch 6/200
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 911us/step - accuracy: 0.8158 - loss: 0.3979
Epoch 7/200
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 915us/step - accuracy: 0.8218 - loss: 0.3873
Epoch 8/200
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 884us/step - accuracy: 0.8269 - loss: 0.3771
Epoch 9/200
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 878us/step - accuracy: 0.8336 - loss: 0.3673
Epoch 10/200
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 823us/step - accuracy: 0.8372 - loss: 0.3577
Epoch 11/200
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 866us/step - accuracy: 0.8427 - loss: 0.3477
Epoch 12/200
120/1

In [27]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'خسارة الاختبار: {test_loss:.4f}')
print(f'دقة الاختبار: {test_accuracy:.4f}')

y_probability = model.predict(X_test, verbose=0)
y_pred = (y_probability >= 0.5)

print()
print('مصفوفة الالتباس (الصفوف: الحقيقة، الأعمدة: التوقع):')
print(confusion_matrix(y_test, y_pred))
print()
print('تقرير التصنيف:')
print(classification_report(y_test, y_pred, digits=3))

خسارة الاختبار: 2.3024
دقة الاختبار: 0.8088

مصفوفة الالتباس (الصفوف: الحقيقة، الأعمدة: التوقع):
[[1772  219]
 [ 259  250]]

تقرير التصنيف:
              precision    recall  f1-score   support

           0      0.872     0.890     0.881      1991
           1      0.533     0.491     0.511       509

    accuracy                          0.809      2500
   macro avg      0.703     0.691     0.696      2500
weighted avg      0.803     0.809     0.806      2500



In [28]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(accuracy, precision, recall, f1, sep='\t')

0.8088	0.5330490405117271	0.4911591355599214	0.5112474437627812
